Ci si collega da Kademlia a IPFS in quanto IPFS usa una DHT basata su Kademlia per trovare i peer che possano fornire un certo contenuto.
## IPFS
**IPFS (InterPlanetary File System)** rappresenta un sistema P2P distribuito **per memorizzare, individuare e recuperare contenuti su larga scala**.  

La "rivoluzione" in IPFS sta nel meccanismo per identificare le risorse: **invece di basarsi su un indirizzo di posizione in un dato server, si segue un ID basato sul contenuto stesso**. 

Nel Web tradizionale un URL come `https://example.com/documento.pdf` indica un server e un percorso presso cui chiedere la risorsa, e il contenuto disponibile in quell'indirizzo può cambiare nel tempo.  
IPFS invece utilizza un **identificatore derivato dal contenuto** per cui la richiesta diventa qualcosa del tipo "dato il contenuto identificato da questo ID, quali peer possono fornirmelo?" --> non è necessario che il contenuto sia sempre preso dallo stesso server.

Basato sul contenuto significa che la risorsa è identificata tramite **impronta crittografica**.  
Per capire in generale il senso di IPFS una buona analogia è *"Web + BitTorrent + Git"*: 
- Web per la pubblicazione e accesso ai contenuti,
- BitTorrent per la segmentazione dei file in blocchi e il download da più peer,
- Git per le strutture basate su hash e supporto al versioning

### Blocchi, CID e Merkle DAG
In IPFS un file può essere suddiviso in piccoli blocchi, detti **chunk**. Ogni blocco è identificato attraverso un **CID (Content Identifier)**, che rappresenta l'hash crittografico del suo contenuto --> **se il contenuto cambia, anche il CID cambia** (se un file viene modificato si passa a una nuova *versione*, il suo CID sarà diverso) 

I blocchi non sono indipendenti, vengono invece collegati tramite riferimenti basati sui loro identificatori formando un **Merkle DAG (Directed Acyclic Graph)**.

es. con un Merkle Tree: abbiamo alla radice una directory con due file. Uno dei due file è suddiviso in tre blocchi di dimensioni diverse, ognuno con il proprio CID.  
La struttura prevede che il nodo che rappresenta la directory contenga i CID dei due file e come proprio identificativo una firma crittografica su questi riferimenti, e il file suddiviso in blocchi contenga i CID dei tre blocchi e di nuovo firma --> in questo modo a partire dalla directory o dai file si può risalire alle parti necessarie per ricostruirlo. 

<img src="imgs/merkle_dag.png" alt="Merkle DAG" width="400"/>

Se cambia anche solo il secondo blocco --> cambia il suo CID --> cambia il nodo del file che contiene quel CID come riferimento e quindi anche il suo identificativo --> il cambiamento si propaga fino alla radice.  
I blocchi rimasti uguali invece conservano i propri CID e possono essere riutilizzati senza doverli riscaricare.

**Perché questa struttura DAG? Quattro vantaggi:**
1. **Verifica dell'integrità**: quando si riceve un blocco, si può ricalcolarne l'hash e confrontarlo con quello atteso nel suo identificatore --> si può verificare se il blocco è stato corrotto o modificato.
2. **Confronto efficiente tra versioni**: per confrontare due strutture non dobbiamo trasferire tutti i dati, ma possiamo invece partire dagli ID delle radici. Se coincidono --> stessa struttura, se differiscono si scende ai livelli inferiori per individuare i blocchi diversi. Il risparmio oltre che nella strategia sta anche nel fatto che **confrontare id di lunghezza fissa è molto più efficiente che trasferire interi file**
3. **Deduplicazione dei dati**: se due file contengono lo stesso blocco, possono riferirsi allo stesso blocco identificato dal suo CID. Ciò permette di mantenere una sola copia del blocco e evitando copie superflue di parti identiche
4. **Download indipendente e parallelo dei blocchi**: i blocchi possono essere recuperati separatamente, anche da peer diversi e in parallelo: la struttura del file contiene infatti i riferimenti ai CID dei blocchi e quindi le informazioni necessarie per organizzarli correttamente.


### Il ruolo della DHT e funzionamento
IPFS usa una variante di Kademlia, adattata alle proprie esigenze, con $k$-bucket e lookup paralleli e asincroni.  
Attenzione ovviamente che **la DHT serve a trovare chi possiede il contenuto, ma non a trasferirlo**: per questo vengono utilizzati **provider record**, ossia informazioni che associano un CID ai peer che effettivamente lo possiedono.  
Quindi in generale il peer che viene trovato tramite lookup nella DHT può essere diverso da quello che conserva effettivamente il blocco (es. il peer A conserva il blocco -> la DHT arriva a scoprire che A è provider del blocco, quindi sta al richiedente contattarlo separatamente per la fase di retrieval).

<img src="imgs/ipfs_dht.png" alt="IPFS DHT" width="400"/>

Il funzionamento può essere distinto in tre momenti:
1. **Pubblicazione**: un peer dispone del contenuto e annuncia nella DHT di poterlo fornire --> l'annuncio rende possibile trovare il provider (non direttamente il contenuto, come detto)
2. **Ricerca del provider**: un altro peer conosce il CID desiderato ed esegue un lookup per scoprire quali peer possono fornirlo. Il risultato è un elenco di peer che hanno annunciato di possedere il contenuto.
3. **Trasferimento**: il richiedente contatta i provider e recupera i blocchi tramite un protocollo retrieval P2P come Bitswap, eventualmente in parallelo.  
Dopo averli ricevuti, ne verifica l'integrità calcolando l'hash e confrontandolo con il CID atteso e infine li ricompone secondo la struttura del Merkle DAG per ottenere il file originale.

In tutto ciò due scenari possibili:
- **Rete pubblica**: i peer partecipano ad una rete condivisa, usata per annunciare e cercare contenuti
- **Rete privata**: IPFS viene usato in un ambiente ristretto, ad es. tra macchine di un'organizzazione --> rete di discovery separata. Come al solito è possibile progettare soluzioni ibride

**Vantaggi**:
- **Decentralizzazione**: il contenuto può essere fornito da più peer, non più dipendenza da un server centrale
- **Integrità e identificazione delle versioni**: i CID permettono di riconoscere il contenuto esatto richiesto
- **Efficienza**: caching, riuso dei blocchi e download in parallelo riducono il traffico e i tempi di recupero
- **Maggiore resistenza alla censura** (sempre decentralizzazione)
- **Accesso locale**: se i contenuti sono disponibili su peer raggiungibili nella LAN --> non è necessario passare per Internet

**Limiti**:
- **Non garantisce automaticamente la conservazione permanente**: sapere che un certo peer offre un contenuto non significa che lo manterrà per sempre. Se nessun peer raggiungibile conserva i blocchi --> siamo fregati
- **I dati che cambiano spesso richiedono più gestione** (ogni modifica produce nuovi CID per le strutture coinvolte --> più versioni da gestire)
- **La ricerca si basa su exact match** in quanto ci si basa su DHT con Kademlia e i CID sono hash crittografici (i protocolli basati su DHT si basano su associazioni chiave-valore... ricerche per argomento, proprietà o intervalli richiedono strumenti aggiuntivi)

Tra i casi d'uso di IPFS la distribuzione di contenuti, applicazioni decentralizzate, dati collegati a blockchain e NFT, condivisione e archiviazione di dati scientifici.  
Una libreria utilizzata da IPFS per la gestione dei blocchi è **libp2p**, open source e framework modulare utile per costruire le comunicazioni P2P.

## Architetture Ibride
Le **architetture ibride combinano elementi di sistemi centralizzati e decentralizzati**. Si vuole sfruttare la semplicità e l'efficienza di alcune funzioni centralizzate, mantenendo però distribuite altre attività come memorizzazione e condivisione dei dati.

Tre esempi: reti con **super-peer**, **BitTorrent con tracker** e **blockchain**.

### Reti con super-peer
Qui alcuni nodi (super-peer) hanno più responsabilità e capacità rispetto agli altri, assumendo il ruolo di **organizzare la ricerca delle risorse per i peer ordinari**. 

Quindi la rete presenta due livelli:
- ogni super-peer mantiene informazioni sulle risorse disponibili presso i peer ordinari che gli sono collegati
- i super peer sono a loro volta collegati tra loro tramite overlay e collaborano per cercare risorse al di fuori del proprio gruppo

In questo senso i super peer svolgono il ruolo di **directory locale**, dal momento che conosce quali risorse sono disponibili nel gruppo senza però che debba conservarle tutte.

<img src="imgs/superpeer.png" alt="Super-peer" width="400"/>

Supponiamo che un nodo ordinario stia cercando un file. La ricerca avviene come segue:
1. contatta il proprio super-peer
2. il super-peer controlla se il file è disponibile presso uno dei peer del gruppo. Se sì restituisce il riferimento al peer che lo possiede, altrimenti estende la ricerca contattando altri super-peer

Qui protocolli adattabili sono anche il **flooding** o la **random-walk**, in quanto stavolta l'overlay di super peer è più piccolo!  
Chiaramente si tratta di un'architettura ibrida perché i super-peer svolgono un ruolo centralizzato di indicizzazione e ricerca.

Tra i problemi da gestire:
- **Come associare i peer ai super-peer** --> o *associazione statica* (peer scelgono un super-peer e lo mantengono) o *associzione dinamica* (un peer può passare a un super-peer più adatto, magari perché più vicino o meno carico)
- **Come selezionare i super-peer** --> qui si possono usare algoritmi di leader ekection, che però considerino le caratteristiche dei candidati (infatti un super-peer deve avere più capacità di calcolo, memoria e banda rispetto a un peer ordinario)



### BitTorrent con tracker
BitTorrent è un sistema P2P per la condivisione di file. Consideriamo ora la versione con **tracker** (esiste anche la modalità *trackless* basata su DHT e derivata da Kademlia).

Il principio fondamentale è che **un file viene suddiviso in blocchi, scaricabili da peer differenti anche in parallelo**.  
Un peer può inoltre condividere blocchi già ricevuti mentre sta ancora scaricando quelli mancanti, non deve aspettare di avere il file completo per contribuire alla distribuzione.

Il **tracker** rappresenta un servizio che *tiene traccia dei peer partecipanti alla condivisione di un certo contenuto*. Questo permette a un nuovo partecipante di ottenere i contatti iniziali nella fase di **bootstrap**.  
Il procedimento è il seguente:
1. **il client ottiene il file .torrent da altri peer**: il file torrent contiene metadati relativi alla condivisione (tra cui il riferimento al tracker). Non è il file che si vuole scaricare, ma un file di piccole dimensioni che contiene informazioni su come ottenere il contenuto desiderato. (il torrent viene tipicamente distribuito tramite siti web o canali di comunicazione esterni dai peer che vogliono condividere il contenuto)
2. **il client contatta il tracker**, che restituisce una lista di peer che partecipano alla condivisione e possono avere il file o alcuni dei suoi blocchi
3. **il client contatta i peer** per scaricare 

Quindi importante che il tracker serve solo a individuare i peer, ma il contenuto effettivo viene trasferito direttamente tra i peer. 

Uno **swarm** rappresenta l'insieme dei peer che partecipano alla condivisione dello stesso contenuto (es. scarica un blocco da un partecipante, scaricarne un altro contemporaneamente da un altro peer e intanto inviare a un terzo peer i blocchi già ricevuti).  
Ciò spiega perché questa versione di BitTorrent è considerata un'architettura ibrida: il tracker svolge un ruolo centralizzato di indicizzazione, mentre il trasferimento dei blocchi avviene in modo distribuito tra i peer.

Non tutti i blocchi sono disponibili sullo stesso numero di peer --> BitTorrent implementa un meccanismo **Rarest Piece First**, per cui si dà priorità di download ai blocchi meno diffusi affinché la disponibilità dei blocchi sia più uniforme.

In un sistema del genere una componente fondamentale è il **meccanismo di incentivi** per evitare che i peer si comportino da **free-rider**, ossia scarichino senza condividere.  
In questo senso BitTorrent implementa il meccanismo di **tit-for-tat**: semplificando, *un peer privilegia l'invio dei dati verso peer dai quali li sta ricevendo*. 

### Blockchain (molto ad alto livello)
Introdotte per via dell'esigenza di **gestire transazioni in modo decentralizzato, verificandone la validità e conservandone una registrazione affidabile**. 

Es. Tx 10 euro da Alice a Bob --> **Validazione** == verificare che la tx rispetti le regole del sistema (es. no double spending)  e **memorizzazione** == conservare la tx in modo persistente per poterla consultare in futuro.

Le domande quindi sono: chi decide che una tx è valida? Dove viene conservata? Come fanno i peer a concordare sulla storia delle transazioni?  
N.B. : una tx non si limita alle criptovalute, ma possono rappresentare più in generale **registrazioni o operazioni relative a documenti, risorse, votazioni e altri dati..**

In generale, una **blockchain** è **un registro di tx organizzate in blocchi connessi tra loro e replicato tra più partecipanti**.  
Il registro in cui sono memorizzate le tx è detto **ledger**, in questo senso la blockchain appartiene alla famiglia delle **DLT (Distributed Ledger Technology)**.  

L'obiettivo è mantenere una storia condivisa e resistente a alterazioni anche quando i nodi sono **untrusted**.  
In questo senso si distingue il concetto di **crash failure** di un nodo (che smette di funzionare) da quello di **Byzantine failure**, ossia un nodo si comporta in modo malevolo al fine di far fallire il sistema. 

La presenza di nodi bizantini (prima introduzione con **Byzantine Generals Problem**, Lamport) introduce il tema di **consenso distribuito** che approfondiremo successivamente.

#### Funzionamento generale e struttura della catena
Si descriverà il funzionamento generale di una blockchain in un modello semplificato e la struttura relativa di ogni blocco (vedi corso Crypto di Pasquale per dettagli seri).  
Si assume che ogni peer mantenga una replica locale del ledger, ossia una copia della storia delle transazioni.

L'algoritmo di consenso prevede in generale che, *quando viene proposta una nuova tx*:
1. la richiesta venga diffusa in broadcast a tutti i peer della rete
2. la tx venga validata per verificarne la correttezza secondo le regole del sistema 
3. le varie tx validate vengano raccolte in un blocco, che a sua volta sarà effettivamente aggiunto alla catena solamente dopo che il blocco è stato validato secondo le regole del consenso.

Le transazioni sono quindi raggruppate in blocchi, ognuno di essi è suddiviso in:
- **header**: contiene informazioni di controllo e collegamento, tra cui principalmente **l'hash dell'header del blocco precedente**, **timestamp**, **hash del body del blocco corrente** e **nonce** legato al meccanismo di consenso (es. proof-of-work)
- **body**: contiene le transazioni validate che il blocco registra, tipicamente organizzate in una struttura Merkle Tree per consentire la verifica dell'integrità dei dati.

Quindi la chain si sviluppa in quanto ogni blocco contiene un riferimento basato sull'hash dell'header del blocco precedente --> se un bizantino tenta di modificare una tx in un blocco già registrato, cambia l'hash dell'header di quel blocco e quindi anche il riferimento nel blocco successivo e così a catena fino all'ultimo blocco --> il nuovo blocco sarà rifiutato.  
Chiaramente questo meccanismo si basa fortemente sul fatto che la funzione hash crittografica sia **one-way** e **collision-resistant**, più precisamente deve essere computazionalmente difficile trovare collisioni che possano risultare  utili a sostituire blocchi e falsificare i dati della catena. 

<img src="imgs/blockchain.png" alt="Blockchain" width="600"/>

**Ma chi decide chi può aggiungere il prossimo blocco della catena?** È necessario un **meccanismo di consenso** che permetta di capire chi, tra i leader eletti (partecipanti quindi al processo di consenso), può contribuire all'allungamento della catena.

La soluzione banale sarebbe affidarsi ad una singola entità fidata, ma questo chiaramente andrebbe contro la volontà iniziale di gestire le transazioni in modo decentralizzato --> due alternative: affidare la decisione ad un gruppo autorizzato o totalmente decentralizzato, con protocollo di consenso basato su tutti i peer.

##### Blockchain permissioned
In questo caso la partecipazione al processo decisionale è riservata a un **gruppo di nodi autorizzati**.  
Nello scenario descritto il gruppo è relativamente piccolo, con membri preselezionati e ammessi secondo specifiche regole e che possono cambiare nel tempo. Sono i membri a raggiungere un accordo su come far avanzare il registro.

Gruppo di nodi autorizzati non significa necessariamente fidati al 100% --> si resta sempre nel mondo untrusted e quindi per cui i nodi possono essere bizantini.  
Si è dimostrato come nel mondo permissioned e asincrono, affinché possa esservi consenso (problema del Byzantine Agreement), il numero di nodi corrotti deve essere meno di un terzo rispetto alla totalità dei nodi:
$$f \le \frac{n-1}{3} == n \ge 3f + 1$$
Tra le blockchain permissioned abbiamo **Hyperledger** o **Corda**.

##### Blockchain permissionless
In questo caso il consenso si basa su un protocollo aperto a tutti i peer della rete. Per garantire sicurezza e resistenza a comportamenti bizantini, si ricorre a meccanismi di consenso come **Proof-of-Work (PoW)** o **Proof-of-Stake (PoS)**.

PoW == competizione abasata su potenza computazionale: i peer competono per risolvere un puzzle crittografico, il primo che lo risolve può proporre il blocco successivo.

PoS == competizione basata su stake: i peer competono per proporre il blocco successivo in base alla quantità di risorse (es. criptovaluta) che hanno messo in gioco.

Vedremo più in dettaglio PoW e PoS successivamente.

Il mondo permissionless non è soggetto al risultato per cui massimo un terzo dei nodi può essere bizantino, ma la sicurezza del sistema dipende da quanto potere computazionale o stake sia controllato da nodi onesti rispetto a quelli malevoli.

È per via delle blockchain permissioned che abbiamo catalogato blockchain tra le architetture ibride: la blockchain permissionless invece è del tutto decentralizzata. Esempi: **Bitcoin**, **Ethereum**.